# Task scope and the limits of learned compression

A quick look over the retained measurements. Reads `results/` through relative
paths, trains nothing, downloads nothing, and runs in a few seconds.

In [ ]:
import csv
import json
from pathlib import Path

import numpy as np

from taskscope.paths import RESULTS

def load_json(*parts):
    return json.loads((RESULTS.joinpath(*parts)).read_text())

def load_csv(*parts):
    with RESULTS.joinpath(*parts).open(newline="") as stream:
        return list(csv.DictReader(stream))

manifest = load_json("MANIFEST.json")
print(f"{manifest['files']} retained files, {manifest['bytes'] / 1e6:.1f} MB")

## Exact rates

Two finite task families. In the first, the reduced observation cannot attain
the additional task's full-observation risk at any rate. In the second, both
observations share one Bayes prediction while assigning different costs to
coding errors.

In [ ]:
summary = load_json("exact", "summary.json")
single = summary["confidence_at_risk_0_30"]
pair = summary["confidence_family_at_risk_0_30"]

print(f"threshold split risk floor        {summary['threshold_split_risk_floor']}")
print(f"threshold zero-allowance gap      {summary['threshold_zero_allowance_baseline_gap_bits']:.5f} bits")
print(f"one task, full observation        {single['source_direct_bits']:.5f} bits")
print(f"one task, reduced observation     {single['split_bits']:.5f} bits")
print(f"complementary pair, both          {pair['source_direct_bits']:.5f} bits")
print(f"second-task risk of that channel  {pair['single_task_channel_second_task_risk']:.5f}")

In [ ]:
checks = {record["name"]: record for record in load_json("exact", "profile_numerical_checks.json")}
print("successive rates as tasks are added to the three-state cyclic family")
for name in ("cyclic_1", "cyclic_12", "cyclic_123"):
    record = checks[name]
    print(f"  {record['tasks']} task(s)  {record['formula_rate']:.6f} bits"
          f"   free joint-action optimum differs by {record['absolute_discrepancy']:.2e}")
print(f"\nall {len(checks)} cases agree with the free optimum to "
      f"{max(r['absolute_discrepancy'] for r in checks.values()):.1e}")

## Learned block codes

Complete message lengths, including indices, headers, and padding.

In [ ]:
rows = load_json("block_coding", "summary.json")

def mean_rate(condition, access, length=16):
    method = "reference" if access == "reduced" else "learned"
    selected = [r for r in rows if r["condition"] == condition and r["length"] == length
                and r["access"] == access and r["method"] == method]
    return float(np.mean([r["complete_rate"] for r in selected]))

print("length-16 mean complete rate, bits per decision")
print(f"{'requirement':<16}{'with confidence':>17}{'without':>12}")
for name, label in [("single", "single"), ("duplicate", "duplicate"),
                    ("loose", "loose pair"), ("pair", "tight pair")]:
    condition = f"two_tau_0_{name}"
    print(f"{label:<16}{mean_rate(condition, 'full'):>17.5f}{mean_rate(condition, 'reduced'):>12.5f}")

In [ ]:
print("mean rate saving from observing confidence, by corruption probability")
for tau, label in [("0", "0"), ("0.125", "1/8"), ("0.5", "1/2")]:
    single = mean_rate(f"two_tau_{tau}_single", "reduced") - mean_rate(f"two_tau_{tau}_single", "full")
    pair = mean_rate(f"two_tau_{tau}_pair", "reduced") - mean_rate(f"two_tau_{tau}_pair", "full")
    print(f"  tau {label:<4} single {single:8.5f}   tight pair {pair:8.5f}")

forecasts = load_json("block_coding", "forecasts.json")
errors = [abs(d) for f in forecasts for d in f["assessment_minus_forecast"]]
agree = sum(f["predicted_feasible"] == f["observed_feasible"] for f in forecasts)
print(f"\n{len(forecasts)} additional-task forecasts, largest component error {max(errors):.6f},"
      f" mean-threshold decisions agreeing {agree}/{len(forecasts)}")

## CIFAR-100

Six selections frozen on validation, then assessed on test. The same
logit-preserving message is read three ways.

In [ ]:
test = load_json("cifar", "test_extension_summary.json")
labels = {17: "A", 23: "B", 29: "C"}

print(f"{'teacher':<9}{'preserve':<10}{'bpp':>8}{'inherited':>11}{'refit tensor':>14}{'refit logits':>14}   choice")
for group in sorted(test["groups"], key=lambda g: (g["teacher_seed"], -g["actual_bpp"])):
    choice = "validation selection" if group["selected_on_validation"] else "lower-rate neighbour"
    print(f"{labels[group['teacher_seed']]:<9}{group['objective']:<10}{group['actual_bpp']:>8.3f}"
          f"{100 * group['fixed_fine_accuracy']:>11.2f}{100 * group['fine_accuracy']:>14.2f}"
          f"{100 * group['logit_fine_accuracy']:>14.2f}   {choice}")

In [ ]:
print("logit minus early, selected pairs")
for comparison in test["comparisons"]:
    low, high = comparison["fine_difference_image_ci95"]
    print(f"  teacher {labels[comparison['teacher_seed']]}"
          f"   fine accuracy {100 * comparison['late_minus_early_fine_accuracy']:+.3f} pt"
          f"   interval [{100 * low:+.3f}, {100 * high:+.3f}]"
          f"   rate saving {100 * comparison['relative_rate_saving']:.2f}%")

## Taskonomy

Twelve codecs under componentwise requirements, then three fits of each focal
setting. Normalized risk at or below the tolerance meets that requirement.

In [ ]:
pool = load_csv("taskonomy", "candidate_pool.csv")
fields = ["actual_bpp", "depth", "semantic", "edge", "semantic_object_miou"]

print("test pool, normalized component risks are not comparable across tasks")
print(f"{'setting':<16}" + "".join(f"{f.replace('semantic_object_', ''):>12}" for f in fields))
for row in pool:
    if row["split"] != "test":
        continue
    print(f"{row['setting']:<16}" + "".join(f"{float(row[f]):>12.5f}" for f in fields))

In [ ]:
tolerances = load_json("taskonomy", "pool", "development_tolerances.json")
print("three-fit focal means on test")
print(f"{'setting':<14}{'bpp':>9}{'depth':>9}{'semantic':>10}{'edge':>9}{'mIoU':>9}"
      f"{'bytes':>9}{'header':>9}")
for row in load_json("taskonomy", "focal_means.json"):
    losses, byte_counts = row["losses"], row["mean_bytes"]
    print(f"{row['setting']:<14}{row['actual_bpp']:>9.5f}{losses['depth']:>9.5f}"
          f"{losses['semantic']:>10.5f}{losses['edge']:>9.5f}{row['mean_original_miou']:>9.5f}"
          f"{byte_counts['bytes']:>9.3f}{row['header_fraction_of_total_bytes']:>9.4f}")

print("\nmixtures selected on validation, applied unchanged on test")
for row in load_csv("taskonomy", "mixture_selections.csv")[:6]:
    print("  " + "  ".join(f"{key}={row[key]}" for key in list(row)[:6]))

## Motivating species pair

Blue Grosbeak is not an ImageNet class and the frozen teacher assigns it the
Indigo Bunting output. Readers fitted on the complete feature keep the species
apart at every residual block, while readers fitted on the RGB reconstruction of
that same feature lose accuracy as the wing cue fades.

In [ ]:
held = load_json("pilot", "pair", "heldout_evaluation.json")
block = held["cub_test"]["restricted_200way"]
exits = ["layer3", "block1", "block2", "block3"]
labels = ["block 13", "block 14", "block 15", "block 16"]

def accuracy(key, fit):
    confusion = np.array(block[key]["candidates"][fit]["restricted"]["confusion"])
    return 100 * np.trace(confusion) / confusion.sum()

print("species-pair accuracy (%), 200-way readers restricted to the two species,")
print("30 official test photographs per species, two fits per observation")
print(f"{'observation':<26}" + "".join(f"{name:>10}" for name in labels))
for name, prefix in (("Native feature", ""), ("Reconstructed RGB", "inverse_")):
    for fit in (0, 1):
        cells = "".join(f"{accuracy(prefix + key, fit):>10.1f}" for key in exits)
        print(f"{name + f', fit {fit + 1}':<26}{cells}")
    means = "".join(f"{np.mean([accuracy(prefix + key, f) for f in (0, 1)]):>10.1f}"
                    for key in exits)
    print(f"{name + ', mean':<26}{means}")

In [ ]:
numbers = load_json("pilot", "pair", "figure_numbers.json")

print(numbers["statistic"])
print(f"readers per observation: {numbers['fitted_readers_per_observation']}")
print(f"test photographs per species: {numbers['test_photographs_per_species']}")
print(f"native ImageNet decision agreement: "
      f"{numbers['imagenet_native_decision_agreement_percent']}%")
print(f"cue colour curve: {numbers['cue_color_curve']}")